# Homework 6

Solution to the sixth homework of the Machine Learning Zoomcamp 2026, in which I explore the car fuel-efficiency dataset and train decision tree regressors, random forest regressors and XGBoost regressors for different parameters.

## Import Libraries and Load the Data

Import NumPy, pandas, scikit-learn and XGBoost, load the dataset and inspect the first rows.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor

import xgboost as xgb

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv")

In [3]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


## Data Exploration and Preparation

Inspect descriptive statistics, check missing values and fill them with 0.

In [4]:
df.describe()

,model_year,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
count,10000.000000,10000.00000,10000.000000,10000.000000,9123.000000,10000.000000,9736.000000,10000.000000
mean,1999.549500,3.50400,2268.807000,6.035200,254.446016,4286.754000,18.888044,29.997700
std,14.207925,0.95209,183.447355,0.542578,22.844613,266.212491,1.279543,2.930245
min,1975.000000,2.00000,1590.000000,4.000000,171.000000,3320.000000,14.000000,19.800000
25%,1987.000000,3.00000,2150.000000,6.000000,239.000000,4110.000000,18.000000,28.000000
50%,2000.000000,4.00000,2270.000000,6.000000,254.000000,4280.000000,18.900000,30.000000
75%,2012.000000,4.00000,2390.000000,6.000000,270.000000,4470.000000,19.700000,31.900000
max,2024.000000,5.00000,3110.000000,8.000000,334.000000,5460.000000,23.800000,41.200000


In [5]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [6]:
df[['horsepower','acceleration']] = df[['horsepower','acceleration']].fillna(0.0)

In [7]:
df.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

## Split the Data

Split the data into training, validation and test datasets.

In [8]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [9]:
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

In [10]:
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']

In [11]:
train_dicts = df_train.to_dict(orient='records')

In [12]:
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)

## Train a Decision Stump

Train a tree with a depth of one and find which feature is used for splitting the data (Q1).

In [13]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=1)

In [14]:
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



The original feature used to split the data is `'model_year'`.

## Train a Random Forest Regressor

Train a random forest regressor and compute its RMSE on the validation data (Q2).

In [15]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

RandomForestRegressor(n_estimators=10, n_jobs=-1, random_state=1)

In [16]:
val_dicts = df_val.to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [17]:
y_pred = rf.predict(X_val)

In [18]:
error = y_pred - y_val
mse = (error ** 2).mean()
round(np.sqrt(mse),3)

1.837

The RMSE of this model on the validation dataset is 1.837.

## Tune the `n_estimators` Parameter

Train random forest estimators for `n_estimators` in `[10, 50, 100, 150]` and see which value gives the lowest validation RMSE (Q3).

In [19]:
rmse_dic = {}

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)

    error = y_pred - y_val
    mse = (error ** 2).mean()

    rmse_dic[n] = round(np.sqrt(mse),3)

In [20]:
rmse_dic

{10: 1.837, 50: 1.772, 100: 1.768, 150: 1.769}

The lowest validation RMSE is obtained for `n_estimators = 100`.

## Tune the `max_depth` Parameter

Train random forest estimators for `max_depth` in `[10, 15, 20, 25]` and `n_estimators` in `[10, 50, 100, 150]` and see which value of `max_depth` gives the lowest mean validation RMSE (Q4).

In [21]:
rmse_dic = {}

for md in [10, 15, 20, 25]:
    l = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=md, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)

        error = y_pred - y_val
        mse = (error ** 2).mean()

        l.append(np.sqrt(mse))

    rmse_dic[md] = np.mean(l)
        

In [22]:
rmse_dic

{10: 1.7554047863550184,
 15: 1.7839526938716292,
 20: 1.788242068569452,
 25: 1.786317363879773}

The best `max_depth` using RMSE is 10.

## Feature Importance

Use a random forest regressor to find feature importance (Q5).

In [23]:
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, n_estimators=10, n_jobs=-1, random_state=1)

In [24]:
importances_dic = {}

for i in range(len(dv.get_feature_names_out())):
    importances_dic[dv.get_feature_names_out()[i]] = rf.feature_importances_[i]

In [25]:
importances_dic

{'acceleration': 0.055201119469467735,
 'drivetrain=All-wheel drive': 0.007681663098158673,
 'drivetrain=Front-wheel drive': 0.020145318325437353,
 'drivetrain=Rear-wheel drive': 0.0057808697447890986,
 'engine_displacement': 0.06100490942209099,
 'fuel_type=Diesel': 0.017038345612737708,
 'fuel_type=Gasoline': 0.16071245499347317,
 'fuel_type=Hybrid': 0.015799723579542093,
 'horsepower': 0.07801732200517689,
 'model_year': 0.2960013907097287,
 'num_cylinders': 0.007283541644733504,
 'num_doors': 0.019414978403743417,
 'origin=Asia': 0.006390907907055008,
 'origin=Europe': 0.003504506701702116,
 'origin=USA': 0.04214437633660758,
 'vehicle_weight': 0.20387857204555615}

The feature `vehicle_weight` is more important than the features `horsepower`, `acceleration` and `engine_displacement`.

## XGBoost Model

Train XGBoost models for `eta` equal to 0.3 and 0.1 and check which `eta` leads to the best RMSE score on the validation dataset (Q6).

In [26]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

In [27]:
watchlist = [(dtrain, 'train'), (dval, 'val')]

In [28]:
xgb_params = {
    'eta': 0.3, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                  verbose_eval=5,
                  evals=watchlist)

[0]	train-rmse:2.40653	val-rmse:2.46782
[5]	train-rmse:1.59328	val-rmse:1.76216
[10]	train-rmse:1.47125	val-rmse:1.73205
[15]	train-rmse:1.41908	val-rmse:1.73512
[20]	train-rmse:1.39368	val-rmse:1.73766
[25]	train-rmse:1.35837	val-rmse:1.74359
[30]	train-rmse:1.33243	val-rmse:1.74548
[35]	train-rmse:1.29290	val-rmse:1.75198
[40]	train-rmse:1.25326	val-rmse:1.76296
[45]	train-rmse:1.21527	val-rmse:1.77244
[50]	train-rmse:1.18645	val-rmse:1.77517
[55]	train-rmse:1.14797	val-rmse:1.78668
[60]	train-rmse:1.10265	val-rmse:1.79401
[65]	train-rmse:1.06590	val-rmse:1.80143
[70]	train-rmse:1.04604	val-rmse:1.80444
[75]	train-rmse:1.02727	val-rmse:1.80682
[80]	train-rmse:0.99487	val-rmse:1.81308
[85]	train-rmse:0.96970	val-rmse:1.81600
[90]	train-rmse:0.94986	val-rmse:1.82135
[95]	train-rmse:0.91805	val-rmse:1.82544
[99]	train-rmse:0.90106	val-rmse:1.82646


In [29]:
xgb_params = {
    'eta': 0.1, 
    'max_depth': 6,
    'min_child_weight': 1,
    
    'objective': 'reg:squarederror',
    'nthread': 8,
    
    'seed': 1,
    'verbosity': 1,
}

model = xgb.train(xgb_params, dtrain, num_boost_round=100,
                  verbose_eval=5,
                  evals=watchlist)

[0]	train-rmse:2.74739	val-rmse:2.78635
[5]	train-rmse:2.12109	val-rmse:2.20428
[10]	train-rmse:1.80592	val-rmse:1.93152
[15]	train-rmse:1.64286	val-rmse:1.80346
[20]	train-rmse:1.55847	val-rmse:1.75076
[25]	train-rmse:1.50918	val-rmse:1.72751
[30]	train-rmse:1.47544	val-rmse:1.71667
[35]	train-rmse:1.44978	val-rmse:1.71287
[40]	train-rmse:1.43074	val-rmse:1.71289
[45]	train-rmse:1.41543	val-rmse:1.71310
[50]	train-rmse:1.40368	val-rmse:1.71358
[55]	train-rmse:1.39009	val-rmse:1.71570
[60]	train-rmse:1.37884	val-rmse:1.71688
[65]	train-rmse:1.37172	val-rmse:1.71812
[70]	train-rmse:1.36283	val-rmse:1.71919
[75]	train-rmse:1.35099	val-rmse:1.72057
[80]	train-rmse:1.34578	val-rmse:1.72100
[85]	train-rmse:1.33480	val-rmse:1.72192
[90]	train-rmse:1.32546	val-rmse:1.72303
[95]	train-rmse:1.32292	val-rmse:1.72324
[99]	train-rmse:1.31192	val-rmse:1.72481


`eta = 0.1` leads to the best RMSE score on the validation dataset.